# Runner — Anatomy-Aware Pose Estimation (STL Training)
Notebook **minimale**: tutto il codice vero sta nei moduli `.py` sul repo GitHub.

**Prima di lanciare:**
1. Settings → **Internet: On**
2. **Add Input** → COCO 2017 Keypoints + OCHuman + dataset `pose-baseline-checkpoint`

In [18]:
# === 1. Codice dal repo GitHub ===
import os, sys

REPO_URL = "https://github.com/flaviomassaroni/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks.git"
REPO_DIR = "/kaggle/working/repo"

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}

mod_dir = None
for root, dirs, files in os.walk(REPO_DIR):
    if ".git" in root: continue
    if "config.py" in files:
        mod_dir = root
        break
if mod_dir:
    sys.path.insert(0, mod_dir)
    print("Moduli:", sorted([f for f in os.listdir(mod_dir) if f.endswith(".py")]))
else:
    print("ERRORE: config.py non trovato!")

fatal: could not create leading directories of '/kaggle/working/repo': Permission denied
ERRORE: config.py non trovato!


In [19]:
import stl, inspect
print("BONE_SCALE" in dir(stl))                    # True solo col file nuovo
print("_logcosh" in dir(stl))                       # idem
src = inspect.getsource(stl.bone_ratio_loss)
print("logcosh" in src and "log-cosh" in src)       # True col nuovo

True
True
True


In [16]:
# === 2. Import + seed + check ===
from config import *
import utils, data, network, train
import evaluation as ev
from stl import SkeletalTopologyLoss
import cv2
cv2.setNumThreads(0)  # Forza OpenCV a non spawnare thread propri

set_seed(SEED)
print("Device:", device)
print("Ambiente:", ENV_NAME)
print("COCO val ann:", COCO_VAL_ANN)

# Listing dataset montati: solo su Kaggle. In locale /kaggle/input non esiste.
if os.path.isdir("/kaggle/input"):
    print("\n/kaggle/input contiene:")
    for d in os.listdir("/kaggle/input"):
        print("  -", d)
else:
    print("\n(locale: /kaggle/input non presente, uso path da config)")

Device: cuda
Ambiente: LOCAL
COCO val ann: /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/datasets/coco2017/annotations/person_keypoints_val2017.json

(locale: /kaggle/input non presente, uso path da config)


In [20]:
# === 3. Dati COCO ===
from torch.utils.data import DataLoader

train_samples = data.build_samples(COCO_TRAIN_ANN, min_keypoints=8)
val_samples   = data.build_samples(COCO_VAL_ANN)
print(f"train: {len(train_samples)} | val: {len(val_samples)}")

train_dataset = data.COCOKeypointsDataset(train_samples, COCO_TRAIN_IMG, INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
val_dataset   = data.COCOKeypointsDataset(val_samples,   COCO_VAL_IMG,   INPUT_SIZE, HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

loading annotations into memory...
Done (t=3.58s)
creating index...
index created!
loading annotations into memory...
Done (t=0.07s)
creating index...
index created!
train: 116021 | val: 6352


In [21]:
import config, importlib
importlib.reload(config)
BEST_PTH = config.BEST_PTH
print("BEST_PTH:", BEST_PTH, "| esiste:", os.path.exists(BEST_PTH))

BEST_PTH: /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/models/best.pth | esiste: True


In [ ]:
if False:
    import torch
    from stl import soft_argmax
    model.eval()
    ratios_all = []
    valids_all = []
    with torch.no_grad():
        for bi,(imgs,hms,w) in enumerate(train_loader):
            if bi>=8: break
            imgs,w = imgs.to(device), w.to(device)
            c = soft_argmax(model(imgs), beta=10.0)
            def blen(c,a,b): return torch.sqrt(((c[:,a]-c[:,b])**2).sum(-1)+1e-6)
            r = blen(c,7,9)/(blen(c,5,7)+1e-6)
            v = (w.squeeze(-1)[:,5]>0)&(w.squeeze(-1)[:,7]>0)&(w.squeeze(-1)[:,9]>0)
            ratios_all.append(r[v].cpu())
    r = torch.cat(ratios_all)
    qs = torch.quantile(r, torch.tensor([0.01,0.05,0.25,0.5,0.75,0.95,0.99]))
    print("quantili forearm/upper_arm (solo validi):")
    for p,q in zip([1,5,25,50,75,95,99], qs): print(f"  p{p:02d}: {q:.3f}")

In [ ]:
# === 3c. Calibrazione lambda gradient-norm (Strada B) ===
# Misura ||grad L_t / d(final_layer)|| per ogni termine NON pesato
# e per la heatmap loss. Lambda calibrati come:
#   lambda_t = STL_TARGET_FRAC * g_hm / (g_t + eps)
# Cosi ogni termine imprime ai pesi una frazione TARGET_FRAC della
# spinta della heatmap loss (influenza equalizzata, non valore equalizzato).
# Ref: Chen et al. GradNorm 2018. Vedi stl.py::calibrate_lambdas per i dettagli.
import torch
from stl import SkeletalTopologyLoss, calibrate_lambdas

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(BEST_PTH, map_location=device))

diag_criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA,
)

# Se COCO train non e'ancora scaricato, campiona dal val_loader (1 riga)
try:
    calib_loader = train_loader
except NameError:
    calib_loader = val_loader
    print("ATTENZIONE: uso val_loader per calibrazione (COCO train non disponibile)")

lam_suggest = calibrate_lambdas(
    diag_criterion, model, calib_loader, device,
    target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True,
)
# calibrate_lambdas stampa norme grad + lambda calibrati.
# lam_suggest dict viene letto dalla cella 4b.


In [ ]:
# === 4a. Training BASELINE (gia' completato — NON eseguire) ===
if False:
    import torch
    NUM_EPOCHS = 30
    model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=True).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[15, 25], gamma=0.1)
    criterion = train.WeightedMSELoss()
    history = train.fit(model, train_loader, val_loader, optimizer, scheduler, criterion, device, NUM_EPOCHS, CHECKPOINT_DIR, resume=True)

In [ ]:
# === DIAGNOSTICA C. Violazioni bone_ratio: dentro o fuori dal gate STL? ===
#
# DOMANDA: l'AVR e' dominato da bone_ratio (0.325). La loss bone spinge ma la
# categoria non scende. Ipotesi: le violazioni coinvolgono keypoint NON annotati
# (v=0), che la STL non vede (maschera su target_weight) ma l'AVR sì (score>=MIN_CONF).
#
# MISURA: sul baseline, per ogni violazione bone_ratio contata come nell'AVR,
# controllo se TUTTI i keypoint coinvolti erano annotati (v>0) o se almeno uno
# era occluso/non annotato (v=0). Se la maggioranza cade fuori dal gate STL,
# il gate disallineato e' la causa provata e l'opzione A e' giustificata.
#
# Nessuna modifica a stl.py. Gira sul val (gia' presente).
import torch
import numpy as np
from evaluation import (run_inference, KP_IDX, SYMMETRIC_BONE_PAIRS,
                        BONE_RATIO_THRESHOLD, MIN_CONF, _dist)

# --- Carico baseline ---
model_base = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model_base.load_state_dict(torch.load(BEST_PTH, map_location=device))
model_base.eval()

# --- Inferenza sul val (coords/scores allineati a val_samples, shuffle=False) ---
print("Inferenza baseline su COCO val...")
_, coords_all, scores_all = run_inference(model_base, val_samples, COCO_VAL_IMG, device)
print(f"Pose analizzate: {len(coords_all)}")

# --- Per ogni posa, GT visibility dei keypoint (v): v>0 = annotato (gate STL) ---
def gt_visibility(sample):
    kp = np.array(sample['keypoints']).reshape(NUM_KEYPOINTS, 3)
    return kp[:, 2]   # [17] vettore di visibility (0,1,2)

# --- Conteggio violazioni bone_ratio, classificate per gate ---
# Replica ESATTA della logica AVR (compute_avr_violations, parte bone_ratio):
#   violazione se entrambe le ossa sx/dx hanno score>=MIN_CONF e max/min>1.5
# Per ognuna, guardo la visibility GT dei 4 keypoint coinvolti.
stats = {
    'tot_violazioni': 0,
    'tutti_annotati': 0,       # tutti e 4 i kp hanno v>0  -> DENTRO il gate STL
    'almeno_uno_occluso': 0,   # almeno 1 kp ha v=0        -> FUORI dal gate STL
}
# Dettaglio per quanti kp occlusi
per_n_occlusi = {0: 0, 1: 0, 2: 0, 3: 0, 4: 0}

for coords, scores, sample in zip(coords_all, scores_all, val_samples):
    v = gt_visibility(sample)

    for (a1, b1), (a2, b2) in SYMMETRIC_BONE_PAIRS:
        i_a1, i_b1 = KP_IDX[a1], KP_IDX[b1]
        i_a2, i_b2 = KP_IDX[a2], KP_IDX[b2]
        idxs = [i_a1, i_b1, i_a2, i_b2]

        # gate AVR: tutti e 4 con score predetto >= MIN_CONF
        if not all(scores[i] >= MIN_CONF for i in idxs):
            continue

        len1 = _dist(coords[i_a1], coords[i_b1])
        len2 = _dist(coords[i_a2], coords[i_b2])
        if min(len1, len2) <= 1e-3:
            continue
        if max(len1, len2) / min(len1, len2) <= BONE_RATIO_THRESHOLD:
            continue  # nessuna violazione

        # --- E' una violazione bone_ratio (criterio AVR) ---
        stats['tot_violazioni'] += 1
        n_occlusi = int(sum(1 for i in idxs if v[i] == 0))
        per_n_occlusi[n_occlusi] += 1
        if n_occlusi == 0:
            stats['tutti_annotati'] += 1
        else:
            stats['almeno_uno_occluso'] += 1

# --- Report ---
tot = max(stats['tot_violazioni'], 1)
frac_fuori = stats['almeno_uno_occluso'] / tot
frac_dentro = stats['tutti_annotati'] / tot

print("\n" + "=" * 60)
print("DIAGNOSTICA GATE — violazioni bone_ratio (criterio AVR)")
print("=" * 60)
print(f"  Violazioni bone_ratio totali:       {stats['tot_violazioni']}")
print(f"  DENTRO il gate STL (tutti v>0):      {stats['tutti_annotati']:>5}  ({frac_dentro:.1%})")
print(f"  FUORI dal gate STL (>=1 occluso):    {stats['almeno_uno_occluso']:>5}  ({frac_fuori:.1%})")
print("-" * 60)
print("  Distribuzione per n. keypoint occlusi (v=0) nella violazione:")
for n in range(5):
    bar = "#" * int(40 * per_n_occlusi[n] / tot)
    print(f"    {n} occlusi: {per_n_occlusi[n]:>5}  {bar}")
print("=" * 60)

# --- Verdetto ---
print("\nVERDETTO:")
if frac_fuori >= 0.60:
    print(f"  {frac_fuori:.0%} delle violazioni e' FUORI dal gate STL.")
    print("  -> Il gate disallineato e' la causa PROVATA: la STL non puo'")
    print("     correggere violazioni su keypoint che non vede.")
    print("  -> Opzione A (gate su score>=MIN_CONF) giustificata.")
elif frac_fuori >= 0.35:
    print(f"  {frac_fuori:.0%} fuori dal gate: causa PARZIALE.")
    print("  -> Il gate contribuisce ma non spiega tutto. Valuta opzione B")
    print("     (unione gate) o ispeziona il sotto-termine simmetria.")
else:
    print(f"  Solo {frac_fuori:.0%} fuori dal gate: il gate NON e' la causa principale.")
    print("  -> Le violazioni sono su kp annotati che la STL gia' vede.")
    print("     Guardare altrove: forse il sotto-termine simmetria sposta le")
    print("     coordinate, o lambda_bone troppo basso per agire in 3 epoche.")

In [ ]:
# === DIAGNOSTICA D. Perche' bone_ratio non scende anche se la loss vede i kp? ===
#
# CONTESTO: il 70% delle violazioni bone_ratio e' su kp ANNOTATI (gate non e' la
# causa). La loss bone e' attiva ma la categoria AVR non cala. Tre ipotesi:
#   (1) lambda_bone troppo debole: violazioni borderline (1.5-1.7) che servirebbe
#       piu' spinta/epoche per spingere sotto 1.5.
#   (2) gap read-out: la loss ottimizza soft-argmax(beta=50), l'AVR misura
#       argmax+subpixel. Se il gap e' grande sui kp che violano, la loss migliora
#       un read-out che la metrica non guarda.
#   (3) media batch: la simmetria e' buona in media, il gradiente non si concentra
#       sui pochi casi che violano.
#
# MISURA 1: distribuzione dei rapporti max/min nelle violazioni (dentro gate).
# MISURA 2: gap euclideo soft-argmax vs decode_heatmaps sui kp coinvolti.
# Nessuna modifica a stl.py. Gira sul val.
import torch
import numpy as np
from torch.utils.data import DataLoader
from evaluation import KP_IDX, SYMMETRIC_BONE_PAIRS, BONE_RATIO_THRESHOLD, MIN_CONF, _dist
from data import COCOEvalDataset
from utils import decode_heatmaps, heatmap_to_original
from stl import soft_argmax

model_base = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model_base.load_state_dict(torch.load(BEST_PTH, map_location=device))
model_base.eval()

# Rifaccio inferenza tenendo SIA i due read-out SIA la GT visibility.
# soft-argmax e' in spazio heatmap; per confrontarlo con decode_heatmaps (anch'esso
# spazio heatmap PRIMA di heatmap_to_original) li confronto in spazio heatmap.
eval_dataset = COCOEvalDataset(val_samples, COCO_VAL_IMG, INPUT_SIZE)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

ratios_violation = []      # rapporti max/min delle violazioni (dentro gate)
gaps_violation_kp = []     # gap read-out (px heatmap) sui kp che violano (annotati)
gaps_all_kp = []           # gap read-out su TUTTI i kp annotati (riferimento)

sample_idx = 0
print("Inferenza baseline (doppio read-out)...")
with torch.no_grad():
    for imgs, image_ids, bboxes in eval_loader:
        imgs = imgs.to(device)
        heatmaps = model_base(imgs)                      # [B,K,H,W]
        coords_arg, scores = decode_heatmaps(heatmaps)   # argmax+subpixel, spazio hm [B,K,2]
        coords_soft = soft_argmax(heatmaps, beta=STL_BETA).cpu().numpy()  # soft, spazio hm

        for i in range(imgs.shape[0]):
            sample = val_samples[sample_idx]; sample_idx += 1
            v = np.array(sample['keypoints']).reshape(NUM_KEYPOINTS, 3)[:, 2]
            ca = coords_arg[i]      # [K,2] argmax
            cs = coords_soft[i]     # [K,2] soft
            sc = scores[i]          # [K]

            # gap read-out su tutti i kp annotati (riferimento)
            for k in range(NUM_KEYPOINTS):
                if v[k] > 0:
                    gaps_all_kp.append(float(np.linalg.norm(ca[k] - cs[k])))

            # violazioni simmetria (criterio AVR) in spazio heatmap-argmax
            for (a1, b1), (a2, b2) in SYMMETRIC_BONE_PAIRS:
                ia1, ib1, ia2, ib2 = KP_IDX[a1], KP_IDX[b1], KP_IDX[a2], KP_IDX[b2]
                idxs = [ia1, ib1, ia2, ib2]
                if not all(sc[j] >= MIN_CONF for j in idxs):
                    continue
                len1 = _dist(ca[ia1], ca[ib1])
                len2 = _dist(ca[ia2], ca[ib2])
                if min(len1, len2) <= 1e-3:
                    continue
                r = max(len1, len2) / min(len1, len2)
                if r <= BONE_RATIO_THRESHOLD:
                    continue
                # violazione. mi interessa il caso DENTRO il gate (tutti annotati)
                if all(v[j] > 0 for j in idxs):
                    ratios_violation.append(r)
                    for j in idxs:
                        gaps_violation_kp.append(float(np.linalg.norm(ca[j] - cs[j])))

ratios_violation = np.array(ratios_violation)
gaps_violation_kp = np.array(gaps_violation_kp)
gaps_all_kp = np.array(gaps_all_kp)

print("\n" + "=" * 60)
print("MISURA 1 — distribuzione rapporti max/min nelle violazioni (gate dentro)")
print("=" * 60)
print(f"  n violazioni: {len(ratios_violation)}")
if len(ratios_violation):
    qs = np.percentile(ratios_violation, [50, 75, 90, 95, 99])
    print(f"  soglia AVR = {BONE_RATIO_THRESHOLD}")
    print(f"  mediana: {qs[0]:.3f}   p75: {qs[1]:.3f}   p90: {qs[2]:.3f}   "
          f"p95: {qs[3]:.3f}   p99: {qs[4]:.3f}")
    print(f"  max: {ratios_violation.max():.3f}")
    # quanti sono "borderline" (1.5-1.7) vs "gravi" (>2.0)
    borderline = int(((ratios_violation > 1.5) & (ratios_violation <= 1.7)).sum())
    medi       = int(((ratios_violation > 1.7) & (ratios_violation <= 2.0)).sum())
    gravi      = int((ratios_violation > 2.0).sum())
    n = len(ratios_violation)
    print(f"  borderline (1.5-1.7): {borderline:>5} ({borderline/n:.1%})")
    print(f"  medi       (1.7-2.0): {medi:>5} ({medi/n:.1%})")
    print(f"  gravi      (>2.0):    {gravi:>5} ({gravi/n:.1%})")

print("\n" + "=" * 60)
print("MISURA 2 — gap read-out soft-argmax vs argmax (px heatmap)")
print("=" * 60)
print(f"  Su TUTTI i kp annotati (riferimento):")
print(f"    mediana: {np.median(gaps_all_kp):.3f}   p95: {np.percentile(gaps_all_kp,95):.3f}")
print(f"  Sui kp che VIOLANO bone_ratio:")
if len(gaps_violation_kp):
    print(f"    mediana: {np.median(gaps_violation_kp):.3f}   "
          f"p95: {np.percentile(gaps_violation_kp,95):.3f}   "
          f"max: {gaps_violation_kp.max():.3f}")

print("\n" + "=" * 60)
print("LETTURA")
print("=" * 60)
if len(ratios_violation):
    if borderline / n >= 0.50:
        print(f"  M1: maggioranza borderline ({borderline/n:.0%} in 1.5-1.7).")
        print("      -> ipotesi (1): serve PIU' spinta. Fix: alza lambda_bone")
        print("         (o STL_TARGET_FRAC) e/o piu' epoche. Cheap da provare.")
    elif gravi / n >= 0.40:
        print(f"  M1: molte violazioni GRAVI ({gravi/n:.0%} >2.0).")
        print("      -> violazioni nette che la loss dovrebbe gia' attaccare.")
        print("         Se non si muovono, sospetta (2) o (3).")
    else:
        print("  M1: distribuzione mista, nessun pattern dominante netto.")
gap_viol = np.median(gaps_violation_kp) if len(gaps_violation_kp) else 0
gap_all = np.median(gaps_all_kp) if len(gaps_all_kp) else 0
if gap_viol > 2 * max(gap_all, 1e-6):
    print(f"  M2: gap read-out sui kp che violano ({gap_viol:.2f}) >> media ({gap_all:.2f}).")
    print("      -> ipotesi (2) CONFERMATA: la loss ottimizza un read-out diverso")
    print("         da quello misurato proprio sui kp critici. Fix: allineare la")
    print("         loss su argmax o alzare beta ancora.")
else:
    print(f"  M2: gap sui kp che violano ({gap_viol:.2f}) ~ media ({gap_all:.2f}).")
    print("      -> ipotesi (2) scartata: il read-out NON e' il problema.")
print("=" * 60)

In [ ]:
# === DIAGNOSTICA E. Le violazioni gravi sono foreshortening REALE o errore? ===
#
# CONTESTO: 43% delle violazioni bone_ratio e' grave (>2.0), su kp annotati,
# read-out allineato. Due ipotesi residue:
#   (3a) FORESHORTENING REALE: un arto puntato verso la camera si proietta corto.
#        ratio sx/dx alto ma "vero" in 2D. La loss simmetria assume co-planarita'
#        (commento stl.py 196-201); se rotta, penalizza una predizione corretta.
#        -> floor strutturale dell'AVR, non correggibile senza 3D.
#   (3b) DILUIZIONE GRADIENTE: poche pose gravi diluite nella media di batch ->
#        spinta debole. -> fix tecnico (weighting sui violatori / lambda piu' alto).
#
# MISURA 1: per ogni violazione grave, l'osso corto e' anche corto vs torso?
# MISURA 2: norma gradiente bone sui soli violatori vs intero batch.
import torch
import numpy as np
from torch.utils.data import DataLoader
from evaluation import KP_IDX, SYMMETRIC_BONE_PAIRS, BONE_RATIO_THRESHOLD, MIN_CONF, _dist
from data import COCOEvalDataset, COCOKeypointsDataset
from utils import decode_heatmaps
from stl import soft_argmax, bone_ratio_loss

model_base = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model_base.load_state_dict(torch.load(BEST_PTH, map_location=device))
model_base.eval()

# Indici torso per la scala
SH_L, SH_R, HIP_L, HIP_R = KP_IDX['left_shoulder'], KP_IDX['right_shoulder'], \
                           KP_IDX['left_hip'], KP_IDX['right_hip']

# ---------------------------------------------------------------------------
# MISURA 1 — foreshortening reale vs errore di stima
# ---------------------------------------------------------------------------
eval_dataset = COCOEvalDataset(val_samples, COCO_VAL_IMG, INPUT_SIZE)
eval_loader = DataLoader(eval_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Per le violazioni GRAVI (>2.0), classifico:
#   - "foreshortening-like": osso corto < 0.5 * osso lungo E osso corto < 0.6*torso_seg
#     atteso -> compatibile con un arto proiettato corto
#   - "errore-like": entrambe le ossa di lunghezza plausibile, ma diverse
n_grave = 0
foreshort_like = 0
errore_like = 0
ratios_grave = []

sample_idx = 0
print("MISURA 1: inferenza per classificare violazioni gravi...")
with torch.no_grad():
    for imgs, image_ids, bboxes in eval_loader:
        imgs = imgs.to(device)
        heatmaps = model_base(imgs)
        coords_arg, scores = decode_heatmaps(heatmaps)  # spazio heatmap
        for i in range(imgs.shape[0]):
            sample = val_samples[sample_idx]; sample_idx += 1
            v = np.array(sample['keypoints']).reshape(NUM_KEYPOINTS, 3)[:, 2]
            ca = coords_arg[i]; sc = scores[i]

            # scala torso in spazio heatmap (per normalizzare le lunghezze)
            if all(v[j] > 0 for j in [SH_L, SH_R, HIP_L, HIP_R]):
                sh_mid = (ca[SH_L] + ca[SH_R]) / 2
                hip_mid = (ca[HIP_L] + ca[HIP_R]) / 2
                torso = np.linalg.norm(sh_mid - hip_mid)
            else:
                torso = None

            for (a1, b1), (a2, b2) in SYMMETRIC_BONE_PAIRS:
                ia1, ib1, ia2, ib2 = KP_IDX[a1], KP_IDX[b1], KP_IDX[a2], KP_IDX[b2]
                idxs = [ia1, ib1, ia2, ib2]
                if not all(sc[j] >= MIN_CONF for j in idxs):
                    continue
                if not all(v[j] > 0 for j in idxs):
                    continue  # solo dentro il gate
                len1 = _dist(ca[ia1], ca[ib1])
                len2 = _dist(ca[ia2], ca[ib2])
                if min(len1, len2) <= 1e-3:
                    continue
                r = max(len1, len2) / min(len1, len2)
                if r <= 2.0:
                    continue  # solo violazioni GRAVI
                n_grave += 1
                ratios_grave.append(r)
                short = min(len1, len2)
                # foreshortening-like: osso corto molto piccolo anche vs torso
                if torso is not None and torso > 1e-3:
                    if short < 0.25 * torso:   # arto plausibilmente proiettato corto
                        foreshort_like += 1
                    else:
                        errore_like += 1
                else:
                    errore_like += 1

print("\n" + "=" * 60)
print("MISURA 1 — natura delle violazioni gravi (ratio > 2.0)")
print("=" * 60)
print(f"  Violazioni gravi totali: {n_grave}")
if n_grave:
    print(f"  Foreshortening-like (osso corto < 0.25*torso): {foreshort_like:>4} ({foreshort_like/n_grave:.1%})")
    print(f"  Errore-like (entrambe le ossa plausibili):     {errore_like:>4} ({errore_like/n_grave:.1%})")

# ---------------------------------------------------------------------------
# MISURA 2 — diluizione del gradiente
# ---------------------------------------------------------------------------
# Confronto ||grad bone_ratio_loss / d(final_layer)|| calcolato su:
#   (a) un batch intero di train
#   (b) solo le pose di quel batch che hanno una violazione bone grave
# Se (b) per-posa >> (a) per-posa, la media batch diluisce il segnale.
print("\nMISURA 2: gradiente bone sui violatori vs batch intero...")
# Uso il VAL come proxy (in locale il train non e' scaricato). La misura e' una
# norma di gradiente su pochi batch: il val va benissimo, non stiamo allenando.
# COCOKeypointsDataset serve perche' qui ci servono le heatmap GT (w mask).
_grad_samples = val_samples
_grad_img_dir = COCO_VAL_IMG
train_ds_for_grad = COCOKeypointsDataset(_grad_samples, _grad_img_dir, INPUT_SIZE,
                                         HEATMAP_SIZE, SIGMA, NUM_KEYPOINTS)
grad_loader = DataLoader(train_ds_for_grad, batch_size=32, shuffle=True, num_workers=2)
ref_params = list(model_base.head.final_layer.parameters())

def grad_norm():
    return sum(float(p.grad.detach().pow(2).sum()) for p in ref_params if p.grad is not None) ** 0.5

def has_grave_violation(coords_np, w_np):
    # coords_np [K,2] spazio heatmap, w_np [K] target_weight
    for (a1, b1), (a2, b2) in SYMMETRIC_BONE_PAIRS:
        ia1, ib1, ia2, ib2 = KP_IDX[a1], KP_IDX[b1], KP_IDX[a2], KP_IDX[b2]
        idxs = [ia1, ib1, ia2, ib2]
        if not all(w_np[j] > 0 for j in idxs):
            continue
        len1 = _dist(coords_np[ia1], coords_np[ib1])
        len2 = _dist(coords_np[ia2], coords_np[ib2])
        if min(len1, len2) <= 1e-3:
            continue
        if max(len1, len2) / min(len1, len2) > 2.0:
            return True
    return False

model_base.train()
g_batch_list, g_viol_list, frac_viol_list = [], [], []
it = iter(grad_loader)
N_GRAD_BATCHES = 6
for _ in range(N_GRAD_BATCHES):
    try:
        imgs, hms, w = next(it)
    except StopIteration:
        break
    imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)

    out = model_base(imgs)
    coords = soft_argmax(out, beta=STL_BETA)
    valid_mask = w.squeeze(-1)

    # (a) gradiente bone su tutto il batch
    model_base.zero_grad(set_to_none=True)
    L_all = bone_ratio_loss(coords, valid_mask)
    L_all.backward(retain_graph=True)
    g_all = grad_norm()
    g_batch_list.append(g_all)

    # identifica le pose con violazione grave
    coords_np = coords.detach().cpu().numpy()
    w_np = valid_mask.detach().cpu().numpy()
    viol_idx = [b for b in range(coords_np.shape[0]) if has_grave_violation(coords_np[b], w_np[b])]
    frac_viol_list.append(len(viol_idx) / coords_np.shape[0])

    # (b) gradiente bone solo sulle pose che violano
    if viol_idx:
        model_base.zero_grad(set_to_none=True)
        sub = torch.tensor(viol_idx, device=device)
        L_viol = bone_ratio_loss(coords[sub], valid_mask[sub])
        L_viol.backward(retain_graph=False)
        g_viol_list.append(grad_norm())
    else:
        g_viol_list.append(0.0)

model_base.zero_grad(set_to_none=True)

print("\n" + "=" * 60)
print("MISURA 2 — diluizione gradiente bone (norma su final_layer)")
print("=" * 60)
g_batch = np.mean(g_batch_list)
g_viol = np.mean([g for g in g_viol_list if g > 0]) if any(g > 0 for g in g_viol_list) else 0
frac_viol = np.mean(frac_viol_list)
print(f"  Frazione media di pose con violazione grave per batch: {frac_viol:.1%}")
print(f"  ||grad bone|| su batch intero:        {g_batch:.4e}")
print(f"  ||grad bone|| sui soli violatori:     {g_viol:.4e}")
if g_batch > 1e-12:
    print(f"  Rapporto violatori/batch:             {g_viol/g_batch:.2f}x")

print("\n" + "=" * 60)
print("LETTURA FINALE")
print("=" * 60)
if n_grave:
    if foreshort_like / n_grave >= 0.40:
        print(f"  (3a) FORESHORTENING: {foreshort_like/n_grave:.0%} delle violazioni gravi e'")
        print("       compatibile con arto proiettato corto. Una quota dell'AVR")
        print("       bone_ratio e' un FLOOR strutturale 2D, non correggibile dalla STL.")
        print("       Conclusione onesta per il paper, non un bug.")
    else:
        print(f"  (3a) Foreshortening minoritario ({foreshort_like/n_grave:.0%}): le violazioni")
        print("       gravi sono per lo piu' errori di stima, in teoria correggibili.")
if g_viol > 2 * max(g_batch, 1e-12):
    print(f"  (3b) DILUIZIONE confermata: gradiente sui violatori {g_viol/g_batch:.1f}x il batch.")
    print("       Fix tecnico: weighting sui violatori o lambda_bone piu' alto.")
else:
    print(f"  (3b) Diluizione modesta ({g_viol/max(g_batch,1e-12):.1f}x): non e' il fattore dominante.")
print("=" * 60)

In [22]:
import stl, inspect
src = inspect.getsource(stl.bone_ratio_loss)
print("8b attiva:", "_logcosh(excess" in src)
print("hinge vecchia presente:", "SYMMETRY_CAP" in src)


8b attiva: False
hinge vecchia presente: True


In [23]:
import inspect, stl
print("hinge attiva:", "SYMMETRY_CAP" in inspect.getsource(stl.bone_ratio_loss))
print("8b residua:", "SYM_SCALE" in dir(stl))


hinge attiva: True
8b residua: False


In [ ]:
# === GRID TUNING L1. LR x boost x warmup (ritocchi 2+3+4), hinge invariata ===
#
# OBIETTIVO: trovare la config L1 il cui MIGLIOR checkpoint batte il riferimento
# (hinge, LR 3e-5, boost 5x, no-warmup -> best AVR 0.2875 a E1). Non tocca stl.py:
# variamo solo la dinamica di training (quando/quanto la spinta bone si applica).
#
# RITOCCHI:
#   2 (LR)     : 3e-5 (ref) vs 1e-5 (passi piu' piccoli, minimo forse piu' tardi/profondo)
#   3 (boost)  : 5x (ref) vs 3x (discesa piu' graduale)
#   4 (warmup) : no (ref) vs si (lambda_bone sale da 0.2x a 1x nelle prime 2 epoche)
#
# Ogni run parte dallo STESSO best.pth e gira N_EPOCHS epoche. Registro la
# traiettoria e il MIGLIOR AVR (qualunque epoca) con la sua AP. Selezione finale
# sul checkpoint migliore (come stabilito dall'ablation: si prende il minimo).
import torch, os, time
from tqdm.auto import tqdm
import evaluation as ev
from stl import SkeletalTopologyLoss, calibrate_lambdas

# ---- riferimento noto ----
REF_AVR = 0.2875
BASE_AP, BASE_AVR = 0.498, 0.306
AP_FLOOR = BASE_AP - 0.012
N_EPOCHS = 3

# ---- definizione griglia (5 config) ----
# warmup: se True, lambda_bone effettivo = base * warmup_factor(epoch)
GRID = [
    {"name": "ref_3e5_b5",    "lr": 3e-5, "boost": 5.0, "warmup": False},
    {"name": "A_1e5_b5",      "lr": 1e-5, "boost": 5.0, "warmup": False},
    {"name": "B_3e5_b3",      "lr": 3e-5, "boost": 3.0, "warmup": False},
    {"name": "C_1e5_b3",      "lr": 1e-5, "boost": 3.0, "warmup": False},
    {"name": "D_3e5_b5_warm", "lr": 3e-5, "boost": 5.0, "warmup": True},
]

def warmup_factor(epoch, n_warm=1):
    # epoch 1 -> 0.5 (warmup), epoch >=2 -> 1.0 (pieno)
    # con N_EPOCHS=3 la rampa su 1 epoca lascia 2 epoche a boost pieno
    if epoch >= n_warm + 1:
        return 1.0
    return 0.5

_best = BEST_PTH
assert os.path.exists(_best), f"best.pth non trovato: {_best}"
print(f"Baseline: {_best} | Ambiente: {ENV_NAME}")
print(f"Riferimento da battere: AVR {REF_AVR} (hinge 3e-5 b5)\n")

# --- calibro UNA volta il lambda_bone di base (uguale per tutte le config) ---
# Il boost moltiplica questo valore; cosi le config sono confrontabili.
_cal_model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
_cal_model.load_state_dict(torch.load(_best, map_location=device))
_cal_crit = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA)
print("Calibrazione lambda di base (una volta, condivisa)...")
calibrate_lambdas(_cal_crit, _cal_model, train_loader, device,
                  target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True)
LAM_BASE = {
    'bone': _cal_crit.lambda_bone, 'angle': _cal_crit.lambda_angle,
    'order': _cal_crit.lambda_order, 'collapse': _cal_crit.lambda_collapse,
}
del _cal_model, _cal_crit
torch.cuda.empty_cache()
print(f"lambda_bone base = {LAM_BASE['bone']:.5f}\n")

GRID_DIR = os.path.abspath(os.path.join(CHECKPOINT_DIR, "..", "grid_tuning"))
os.makedirs(GRID_DIR, exist_ok=True)

results = []   # raccolgo il riassunto per la tabella finale

for cfg in GRID:
    print("=" * 70)
    print(f"CONFIG {cfg['name']}: LR={cfg['lr']:.0e} boost={cfg['boost']}x "
          f"warmup={cfg['warmup']}")
    print("=" * 70)
    out_dir = os.path.join(GRID_DIR, cfg['name'])
    os.makedirs(out_dir, exist_ok=True)

    model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
    model.load_state_dict(torch.load(_best, map_location=device))

    criterion = SkeletalTopologyLoss(
        heatmap_criterion=train.WeightedMSELoss(),
        lambda_bone=LAM_BASE['bone'], lambda_angle=LAM_BASE['angle'],
        lambda_order=LAM_BASE['order'], lambda_collapse=LAM_BASE['collapse'],
        beta=STL_BETA)

    optimizer = torch.optim.Adam(model.parameters(), lr=cfg['lr'])

    cfg_hist = []
    best_avr_cfg = (1e9, None, None)  # (avr, epoch, ap)

    for epoch in range(1, N_EPOCHS + 1):
        # applica warmup (o boost pieno) al lambda_bone
        wf = warmup_factor(epoch) if cfg['warmup'] else 1.0
        criterion.lambda_bone = LAM_BASE['bone'] * cfg['boost'] * wf

        t0 = time.time()
        model.train()
        sums = {k: 0.0 for k in ['heatmap','bone','total']}
        for imgs, hms, w in tqdm(train_loader, desc=f"{cfg['name']} e{epoch}", leave=False):
            imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss, terms = criterion(out, hms, w)
            loss.backward()
            optimizer.step()
            for k in sums: sums[k] += terms[k] * imgs.size(0)
        n = len(train_loader.dataset)
        for k in sums: sums[k] /= n

        torch.save(model.state_dict(), f"{out_dir}/epoch_{epoch:02d}.pth")

        model.eval()
        coco_eval, avr = ev.evaluate_on_coco_val(
            model, val_samples, device, results_path=f"{out_dir}/pred_e{epoch:02d}.json")
        ap = float(coco_eval.stats[0]); avr_rate = avr['AVR_pose_rate']
        bone = avr['per_category']['bone_ratio']

        flag = ""
        if avr_rate < best_avr_cfg[0] and ap >= AP_FLOOR:
            best_avr_cfg = (avr_rate, epoch, ap); flag = " *best"
        print(f"  e{epoch} lam_bone={criterion.lambda_bone:.4f} | AP {ap:.4f} "
              f"AVR {avr_rate:.4f} bone {bone:.4f} | {time.time()-t0:.0f}s{flag}")
        cfg_hist.append({'epoch': epoch, 'AP': ap, 'AVR': avr_rate, 'bone': bone})

    results.append({'cfg': cfg, 'best': best_avr_cfg, 'hist': cfg_hist})
    del model, criterion, optimizer
    torch.cuda.empty_cache()
    print()

# ===================== TABELLA FINALE =====================
print("\n" + "=" * 70)
print("RIEPILOGO GRIGLIA — ordinato per miglior AVR")
print("=" * 70)
print(f"{'config':>16} {'LR':>6} {'boost':>6} {'warm':>5}  "
      f"{'best AVR':>9} {'@ep':>4} {'AP@best':>8}  vs ref")
print("-" * 70)
ranked = sorted(results, key=lambda r: r['best'][0])
for r in ranked:
    c = r['cfg']; avr, ep, ap = r['best']
    delta = avr - REF_AVR
    mark = "  <-- batte ref" if avr < REF_AVR else ""
    print(f"{c['name']:>16} {c['lr']:>6.0e} {c['boost']:>5.1f}x "
          f"{str(c['warmup']):>5}  {avr:>9.4f} {ep:>4} {ap:>8.4f}  {delta:+.4f}{mark}")
print("-" * 70)
print(f"  riferimento hinge 3e-5 b5: AVR {REF_AVR} @E1")
winner = ranked[0]
if winner['best'][0] < REF_AVR:
    c = winner['cfg']
    print(f"\n  VINCITORE: {c['name']} -> AVR {winner['best'][0]:.4f} "
          f"(ep {winner['best'][1]}, AP {winner['best'][2]:.4f})")
    print(f"  Guadagno su ref: {winner['best'][0]-REF_AVR:+.4f}. Valuta questo")
    print(f"  checkpoint su OCHuman per confermare la generalizzazione.")
else:
    print(f"\n  Nessuna config batte il riferimento (0.2875). La hinge 3e-5 b5 a E1")
    print(f"  resta la migliore: il tuning della dinamica non aiuta. Passa a #6/#5.")
print("=" * 70)

In [24]:
# === CONFIG A ESTESA. LR 1e-5, boost 5x, 8 epoche, DETERMINISTICA ===
#
# La griglia ha mostrato che A (LR basso) ha il minimo a E3 SENZA rimbalzo:
# segnale che con LR 1e-5 la discesa non e' finita in 3 epoche. Qui la estendo
# a 8 epoche per trovare il vero minimo. Run DETERMINISTICO (seed + num_workers=0
# + generator seedato) cosi i numeri sono riproducibili e un miglioramento e'
# reale, non rumore (la griglia aveva ref=0.2993 vs run originale 0.2875: rumore).
import torch, os, time
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import evaluation as ev
from stl import SkeletalTopologyLoss, calibrate_lambdas

# ---- parametri config A ----
LR_A = 1e-5
BOOST_A = 5.0
N_EPOCHS = 8
REF_AVR_E1 = 0.2875   # miglior risultato storico (run originale, hinge 3e-5 b5)

# ---- DETERMINISMO ----
set_seed(SEED)
g = torch.Generator(); g.manual_seed(SEED)
# train_dataset gia' esiste dalla cella 4; ricreo solo il loader deterministico
train_loader_det = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=0,
    pin_memory=True, generator=g)
print(f"Loader deterministico: {len(train_dataset)} esempi, num_workers=0")

_best = BEST_PTH
assert os.path.exists(_best), f"best.pth non trovato: {_best}"
print(f"Baseline: {_best} | Ambiente: {ENV_NAME}")

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(_best, map_location=device))
print("Pesi baseline caricati")

# ---- calibrazione (deterministica: stesso seed, loader det) ----
criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA)
print("Calibrazione lambda...")
calibrate_lambdas(criterion, model, train_loader_det, device,
                  target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True)
lam_bone_calib = criterion.lambda_bone
criterion.lambda_bone = lam_bone_calib * BOOST_A
print(f"lambda_bone: {lam_bone_calib:.5f} -> {criterion.lambda_bone:.5f} ({BOOST_A}x)\n")

optimizer = torch.optim.Adam(model.parameters(), lr=LR_A)
optimizer.zero_grad(set_to_none=True)

OUT_DIR = os.path.abspath(os.path.join(CHECKPOINT_DIR, "..", "configA_ext8"))
os.makedirs(OUT_DIR, exist_ok=True)

# baseline E0
model.eval()
_, avr0 = ev.evaluate_on_coco_val(model, val_samples, device,
                                  results_path=f"{OUT_DIR}/pred_e00.json")
print(f"{'ep':>3} {'AP':>7} {'AVR':>7} {'bone':>7} {'angle':>7} {'colla':>7}  "
      f"{'L_bone':>8} {'L_hm':>8}  time  note")
print("-" * 84)
pc0 = avr0['per_category']
print(f"{'0':>3} {'-':>7} {avr0['AVR_pose_rate']:>7.4f} "
      f"{pc0['bone_ratio']:>7.4f} {pc0['joint_angle']:>7.4f} {pc0['collapse']:>7.4f}   (baseline E0)")

best = (1e9, None, None)   # (avr, epoch, ap)
history = []
for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    model.train()
    sums = {k: 0.0 for k in ['heatmap','bone','total']}
    for imgs, hms, w in tqdm(train_loader_det, desc=f"A e{epoch}", leave=False):
        imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss, terms = criterion(out, hms, w)
        loss.backward()
        optimizer.step()
        for k in sums: sums[k] += terms[k] * imgs.size(0)
    n = len(train_loader_det.dataset)
    for k in sums: sums[k] /= n

    torch.save(model.state_dict(), f"{OUT_DIR}/epoch_{epoch:02d}.pth")

    model.eval()
    coco_eval, avr = ev.evaluate_on_coco_val(
        model, val_samples, device, results_path=f"{OUT_DIR}/pred_e{epoch:02d}.json")
    ap = float(coco_eval.stats[0]); pc = avr['per_category']; avr_rate = avr['AVR_pose_rate']

    note = ""
    if avr_rate < best[0] and ap >= 0.486:
        best = (avr_rate, epoch, ap); note = " *best"
    if avr_rate < REF_AVR_E1:
        note += " <BATTE STORICO"
    print(f"{epoch:>3} {ap:>7.4f} {avr_rate:>7.4f} "
          f"{pc['bone_ratio']:>7.4f} {pc['joint_angle']:>7.4f} {pc['collapse']:>7.4f}  "
          f"{sums['bone']:>8.4f} {sums['heatmap']:>8.4f}  {time.time()-t0:.0f}s{note}")
    history.append({'epoch': epoch, 'AP': ap, 'AVR': avr_rate, 'bone': pc['bone_ratio']})

# ---- verdetto sulla traiettoria ----
avr_traj = [avr0['AVR_pose_rate']] + [h['AVR'] for h in history]
print("\n" + "=" * 70)
print("VERDETTO — config A estesa (LR 1e-5, boost 5x, 8 epoche)")
print("=" * 70)
print(f"  AVR: " + " -> ".join(f"{a:.4f}" for a in avr_traj))
print(f"  miglior AVR: {best[0]:.4f} @E{best[1]} (AP {best[2]:.4f})")
print(f"  storico (run originale): {REF_AVR_E1}")
print("-" * 70)
# il minimo e' interno (non all'ultima epoca) o la curva sta ancora scendendo?
last_is_min = (best[1] == N_EPOCHS)
if best[0] < REF_AVR_E1:
    print(f"  BATTE lo storico di {best[0]-REF_AVR_E1:+.4f}. Vincitore.")
    if last_is_min:
        print(f"  Minimo all'ULTIMA epoca: potrebbe scendere ancora -> prova +epoche.")
    print(f"  -> valuta epoch_{best[1]:02d}.pth su OCHuman per confermare.")
elif last_is_min:
    print(f"  Minimo all'ULTIMA epoca ({best[0]:.4f}): la curva sta ANCORA scendendo.")
    print(f"  Non ha toccato il fondo in 8 epoche -> estendi ancora o aumenta LR un po'.")
else:
    print(f"  Minimo interno a E{best[1]} ({best[0]:.4f}), poi rimbalza. Questo e' il")
    print(f"  fondo per LR 1e-5. Non batte lo storico 0.2875: il rumore della run")
    print(f"  originale spiega il gap. Config A e' comunque la migliore stabile.")
print("=" * 70)

Loader deterministico: 116021 esempi, num_workers=0
Baseline: /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/models/best.pth | Ambiente: LOCAL
Pesi baseline caricati
Calibrazione lambda...
Calibrazione lambda (gradient-norm, rho=0.1, 4 batch, ref=final_layer):
  norme grad grezze: heatmap=2.103e-04  bone=1.856e-01  angle=2.409e-02  order=2.501e+00  collapse=4.987e-04
  lambda calibrati : bone=0.00011  angle=0.00087  order=0.00010  collapse=0.00194
lambda_bone: 0.00011 -> 0.00057 (5.0x)



inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.11s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.84s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.498
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.777
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.532
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.483
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.523
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.539
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.800
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.576
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | ma

A e1:   0%|          | 0/3626 [00:00<?, ?it/s]

inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

...

inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.80s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.499
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.785
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.531
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.482
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.523
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.538
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.800
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.574
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | ma

A e8:   0%|          | 0/3626 [00:00<?, ?it/s]

inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.79s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.497
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.777
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.528
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.481
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.521
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.538
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.800
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.571
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | ma

In [25]:
del model, optimizer, criterion
torch.cuda.empty_cache()

In [26]:
# === CONFIG A+WARMUP ESTESA. LR 1e-5, boost 5x, WARMUP, 8 epoche, DETERMINISTICA ===
#
# La griglia ha mostrato che A (LR basso) e D (warmup) aiutano ENTRAMBI, leve
# indipendenti (passo vs avvio). Qui le combino: LR 1e-5 + boost 5x + warmup,
# 8 epoche deterministiche. Ipotesi: warmup (avvio morbido) e LR basso (passi
# piccoli) sono complementari -> discesa controllata che non scavalca il minimo.
# Confronta con configA_ext8 (A puro) e col ref della griglia.
# Warmup: lambda_bone effettivo = base*boost*warmup_factor(epoch); rampa su 2
# epoche (piu' lunga di quella della griglia perche' qui abbiamo 8 epoche).
import torch, os, time
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import evaluation as ev
from stl import SkeletalTopologyLoss, calibrate_lambdas

# ---- parametri config A+warmup ----
LR_A = 1e-5
BOOST_A = 5.0
N_EPOCHS = 8
REF_AVR_E1 = 0.2875   # miglior risultato storico (run originale, hinge 3e-5 b5)

def warmup_factor(epoch, n_warm=2):
    # rampa su 2 epoche: E1 -> 0.4, E2 -> 0.7, E3+ -> 1.0 (boost pieno).
    # warmup attivo: lambda_bone effettivo = base * boost * warmup_factor(epoch)
    if epoch >= n_warm + 1:
        return 1.0
    return 0.4 + 0.3 * (epoch - 1)

# ---- DETERMINISMO ----
set_seed(SEED)
g = torch.Generator(); g.manual_seed(SEED)
# train_dataset gia' esiste dalla cella 4; ricreo solo il loader deterministico
train_loader_det = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=0,
    pin_memory=True, generator=g)
print(f"Loader deterministico: {len(train_dataset)} esempi, num_workers=0")

_best = BEST_PTH
assert os.path.exists(_best), f"best.pth non trovato: {_best}"
print(f"Baseline: {_best} | Ambiente: {ENV_NAME}")

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(_best, map_location=device))
print("Pesi baseline caricati")

# ---- calibrazione (deterministica: stesso seed, loader det) ----
criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA)
print("Calibrazione lambda...")
calibrate_lambdas(criterion, model, train_loader_det, device,
                  target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True)
lam_bone_calib = criterion.lambda_bone   # valore base; il warmup*boost si applica nel loop
print(f"lambda_bone calibrato (base) = {lam_bone_calib:.5f}; "
      f"target a regime = {lam_bone_calib*BOOST_A:.5f} ({BOOST_A}x dopo warmup)\n")

optimizer = torch.optim.Adam(model.parameters(), lr=LR_A)
optimizer.zero_grad(set_to_none=True)

OUT_DIR = os.path.abspath(os.path.join(CHECKPOINT_DIR, "..", "configA_warmup_ext8"))
os.makedirs(OUT_DIR, exist_ok=True)

# baseline E0
model.eval()
_, avr0 = ev.evaluate_on_coco_val(model, val_samples, device,
                                  results_path=f"{OUT_DIR}/pred_e00.json")
print(f"{'ep':>3} {'AP':>7} {'AVR':>7} {'bone':>7} {'angle':>7} {'colla':>7}  "
      f"{'L_bone':>8} {'L_hm':>8}  time  note")
print("-" * 84)
pc0 = avr0['per_category']
print(f"{'0':>3} {'-':>7} {avr0['AVR_pose_rate']:>7.4f} "
      f"{pc0['bone_ratio']:>7.4f} {pc0['joint_angle']:>7.4f} {pc0['collapse']:>7.4f}   (baseline E0)")

best = (1e9, None, None)   # (avr, epoch, ap)
history = []
for epoch in range(1, N_EPOCHS + 1):
    # WARMUP: lambda_bone sale gradualmente (E1=0.4x, E2=0.7x, E3+=1.0x del target)
    wf = warmup_factor(epoch)
    criterion.lambda_bone = lam_bone_calib * BOOST_A * wf

    t0 = time.time()
    model.train()
    sums = {k: 0.0 for k in ['heatmap','bone','total']}
    for imgs, hms, w in tqdm(train_loader_det, desc=f"A+warm e{epoch}", leave=False):
        imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss, terms = criterion(out, hms, w)
        loss.backward()
        optimizer.step()
        for k in sums: sums[k] += terms[k] * imgs.size(0)
    n = len(train_loader_det.dataset)
    for k in sums: sums[k] /= n

    torch.save(model.state_dict(), f"{OUT_DIR}/epoch_{epoch:02d}.pth")

    model.eval()
    coco_eval, avr = ev.evaluate_on_coco_val(
        model, val_samples, device, results_path=f"{OUT_DIR}/pred_e{epoch:02d}.json")
    ap = float(coco_eval.stats[0]); pc = avr['per_category']; avr_rate = avr['AVR_pose_rate']

    note = ""
    if avr_rate < best[0] and ap >= 0.486:
        best = (avr_rate, epoch, ap); note = " *best"
    if avr_rate < REF_AVR_E1:
        note += " <BATTE STORICO"
    print(f"{epoch:>3} {ap:>7.4f} {avr_rate:>7.4f} "
          f"{pc['bone_ratio']:>7.4f} {pc['joint_angle']:>7.4f} {pc['collapse']:>7.4f}  "
          f"{sums['bone']:>8.4f} {sums['heatmap']:>8.4f}  "
          f"[lam_b={criterion.lambda_bone:.4f} wf={wf:.1f}] {time.time()-t0:.0f}s{note}")
    history.append({'epoch': epoch, 'AP': ap, 'AVR': avr_rate, 'bone': pc['bone_ratio']})

# ---- verdetto sulla traiettoria ----
avr_traj = [avr0['AVR_pose_rate']] + [h['AVR'] for h in history]
print("\n" + "=" * 70)
print("VERDETTO — config A+warmup estesa (LR 1e-5, boost 5x, warmup, 8 epoche)")
print("=" * 70)
print(f"  AVR: " + " -> ".join(f"{a:.4f}" for a in avr_traj))
print(f"  miglior AVR: {best[0]:.4f} @E{best[1]} (AP {best[2]:.4f})")
print(f"  storico (run originale): {REF_AVR_E1}")
print("-" * 70)
# il minimo e' interno (non all'ultima epoca) o la curva sta ancora scendendo?
last_is_min = (best[1] == N_EPOCHS)
if best[0] < REF_AVR_E1:
    print(f"  BATTE lo storico di {best[0]-REF_AVR_E1:+.4f}. Vincitore.")
    if last_is_min:
        print(f"  Minimo all'ULTIMA epoca: potrebbe scendere ancora -> prova +epoche.")
    print(f"  -> valuta epoch_{best[1]:02d}.pth su OCHuman per confermare.")
elif last_is_min:
    print(f"  Minimo all'ULTIMA epoca ({best[0]:.4f}): la curva sta ANCORA scendendo.")
    print(f"  Non ha toccato il fondo in 8 epoche -> estendi ancora o aumenta LR un po'.")
else:
    print(f"  Minimo interno a E{best[1]} ({best[0]:.4f}), poi rimbalza. Questo e' il")
    print(f"  fondo per LR 1e-5. Non batte lo storico 0.2875: il rumore della run")
    print(f"  originale spiega il gap. Config A e' comunque la migliore stabile.")
print("=" * 70)

Loader deterministico: 116021 esempi, num_workers=0
Baseline: /home/flavio/Documents/University/Computer vision/Anatomy-Aware-Pose-Estimation-on-Lightweight-Networks/anatomy-aware-pose/models/best.pth | Ambiente: LOCAL
Pesi baseline caricati
Calibrazione lambda...
Calibrazione lambda (gradient-norm, rho=0.1, 4 batch, ref=final_layer):
  norme grad grezze: heatmap=2.103e-04  bone=1.856e-01  angle=2.409e-02  order=2.501e+00  collapse=4.987e-04
  lambda calibrati : bone=0.00011  angle=0.00087  order=0.00010  collapse=0.00194
lambda_bone calibrato (base) = 0.00011; target a regime = 0.00057 (5.0x dopo warmup)



inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Loading and preparing results...
DONE (t=30.74s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.81s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.498
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.777
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.532
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.483
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.523
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.539
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.800
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.576
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | m

A+warm e1:   0%|          | 0/3626 [00:00<?, ?it/s]

inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

...

inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.79s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.499
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.785
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.531
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.482
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.524
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.538
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.800
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.574
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | ma

A+warm e8:   0%|          | 0/3626 [00:00<?, ?it/s]

inferenza:   0%|          | 0/199 [00:00<?, ?it/s]

loading annotations into memory...
Done (t=0.09s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.09s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *keypoints*
DONE (t=0.87s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.498
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.777
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.531
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets= 20 ] = 0.484
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets= 20 ] = 0.522
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 20 ] = 0.538
 Average Recall     (AR) @[ IoU=0.50      | area=   all | maxDets= 20 ] = 0.800
 Average Recall     (AR) @[ IoU=0.75      | area=   all | maxDets= 20 ] = 0.574
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | ma

In [ ]:
# === FIX L1. Alza lambda_bone (test diluizione) + sanity 3 epoche ===
#
# DIAGNOSI: bone_ratio e' la categoria che domina l'AVR e non scende. Il gradiente
# sui violatori e' 4.3x il batch ma viene mediato via (diluizione). Fix piu'
# economico: aumentare la spinta del solo termine bone. Qui moltiplico il
# lambda_bone CALIBRATO per BONE_BOOST, lasciando gli altri invariati.
# Test rapido, nessuna modifica a stl.py.
#
# Cosa guardare: la colonna bone_ratio deve SCENDERE attraverso le 3 epoche.
# Se scende con AP intatto -> diluizione confermata, alza ancora o lancia il
# run completo. Se NON scende nemmeno con boost alto -> il residuo e' il floor
# da foreshortening (serve L2/L3).
import torch, os, time
from tqdm.auto import tqdm
import evaluation as ev
from stl import SkeletalTopologyLoss, calibrate_lambdas

# --- PARAMETRO DA VARIARE: prova 5.0, poi 10.0, poi 20.0 ---
BONE_BOOST = 5.0

BASE_AP, BASE_AVR = 0.498, 0.306
AP_FLOOR = BASE_AP - 0.010
N_EPOCHS = 3

_best = BEST_PTH
assert os.path.exists(_best), f"best.pth non trovato: {_best}"
print(f"Baseline: {_best} | Ambiente: {ENV_NAME} | BONE_BOOST = {BONE_BOOST}x")

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(_best, map_location=device))
print("Pesi baseline caricati")

LR_STL = STL_FINE_TUNE_LR
criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA,
)

# calibrazione standard
print("Calibrazione lambda...")
calibrate_lambdas(criterion, model, train_loader, device,
                  target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True)

# --- BOOST: alzo SOLO bone ---
lam_bone_calib = criterion.lambda_bone
criterion.lambda_bone = lam_bone_calib * BONE_BOOST
print(f"\nBOOST: lambda_bone {lam_bone_calib:.5f} -> {criterion.lambda_bone:.5f} ({BONE_BOOST}x)")
print(f"Lambda finali: bone={criterion.lambda_bone:.5f} angle={criterion.lambda_angle:.5f} "
      f"order={criterion.lambda_order:.5f} collapse={criterion.lambda_collapse:.5f}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR_STL)
optimizer.zero_grad(set_to_none=True)

OUT_DIR = os.path.abspath(os.path.join(CHECKPOINT_DIR, "..", f"checkpoints_8b_s07_boost{int(BONE_BOOST)}"))
os.makedirs(OUT_DIR, exist_ok=True)

# baseline per-categoria (E0)
model.eval()
_, avr0 = ev.evaluate_on_coco_val(model, val_samples, device,
                                  results_path=f"{OUT_DIR}/pred_e00.json")
pc0 = avr0['per_category']

print(f"\n{'ep':>3} {'AP':>7} {'AVR':>7} {'bone':>7} {'angle':>7} {'colla':>7}  "
      f"{'L_bone':>8} {'L_ang':>8} {'L_ord':>8} {'L_col':>8}  time")
print("-" * 90)
print(f"{'0':>3} {'-':>7} {avr0['AVR_pose_rate']:>7.4f} "
      f"{pc0['bone_ratio']:>7.4f} {pc0['joint_angle']:>7.4f} {pc0['collapse']:>7.4f}   (baseline)")

history = []
for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    model.train()
    sums = {k: 0.0 for k in ['heatmap','bone','angle','order','collapse','total']}
    for imgs, hms, w in tqdm(train_loader, desc=f"ep {epoch}", leave=False):
        imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss, terms = criterion(out, hms, w)
        loss.backward()
        optimizer.step()
        for k in sums: sums[k] += terms[k] * imgs.size(0)
    n = len(train_loader.dataset)
    for k in sums: sums[k] /= n

    torch.save(model.state_dict(), f"{OUT_DIR}/epoch_{epoch:02d}.pth")

    model.eval()
    coco_eval, avr = ev.evaluate_on_coco_val(model, val_samples, device,
                                             results_path=f"{OUT_DIR}/pred_e{epoch:02d}.json")
    ap = float(coco_eval.stats[0]); pc = avr['per_category']; avr_rate = avr['AVR_pose_rate']

    print(f"{epoch:>3} {ap:>7.4f} {avr_rate:>7.4f} "
          f"{pc['bone_ratio']:>7.4f} {pc['joint_angle']:>7.4f} {pc['collapse']:>7.4f}  "
          f"{sums['bone']:>8.4f} {sums['angle']:>8.4f} {sums['order']:>8.4f} {sums['collapse']:>8.4f}  "
          f"{time.time()-t0:.0f}s")
    history.append({'epoch': epoch, 'AP': ap, 'AVR': avr_rate, 'bone': pc['bone_ratio']})

# --- Verdetto ---
bone_traj = [pc0['bone_ratio']] + [h['bone'] for h in history]
avr_traj = [avr0['AVR_pose_rate']] + [h['AVR'] for h in history]
ap_traj = [h['AP'] for h in history]
print("\n" + "=" * 60)
print(f"VERDETTO (BONE_BOOST = {BONE_BOOST}x)")
print("=" * 60)
print(f"  bone_ratio: " + " -> ".join(f"{b:.4f}" for b in bone_traj))
print(f"  AVR total : " + " -> ".join(f"{a:.4f}" for a in avr_traj))
print(f"  AP        : baseline {BASE_AP:.4f}, poi " + " -> ".join(f"{a:.4f}" for a in ap_traj))
print("-" * 60)
bone_down = bone_traj[-1] < bone_traj[0]
ap_ok = all(a >= AP_FLOOR for a in ap_traj)
print(f"  bone_ratio in calo: {'PASS' if bone_down else 'FAIL'}  "
      f"({bone_traj[-1]-bone_traj[0]:+.4f})")
print(f"  AP regge (>= {AP_FLOOR:.4f}): {'PASS' if ap_ok else 'FAIL'}")
print("-" * 60)
if bone_down and ap_ok:
    print(f"  ESITO: la spinta in piu' MUOVE bone_ratio con AP intatto.")
    print(f"         Diluizione confermata come causa correggibile. Prova ad")
    print(f"         alzare BONE_BOOST ancora, o lancia il run completo (10 ep).")
elif not bone_down:
    print(f"  ESITO: bone_ratio NON scende nemmeno a {BONE_BOOST}x.")
    print(f"         La diluizione non e' il fattore dominante: il residuo e' il")
    print(f"         FLOOR da foreshortening. Serve L2 (violator weighting) o L3.")
else:
    print(f"  ESITO: bone scende ma AP cede. Boost troppo alto: riduci BONE_BOOST.")
print("=" * 60)

In [ ]:
# === CONFERMA E1. Baseline vs STL (epoch_01, L1 boost5) su COCO + OCHuman ===
#
# Consolida il candidato migliore trovato con L1: epoch_01 aveva AVR 0.2875
# (< baseline 0.306) con AP intatta sul sanity. Qui lo valuto in modo COMPLETO:
#   - COCO val: conferma che il fix non ha rotto il dominio di training
#   - OCHuman ZERO-SHOT: e' la tesi del paper -> i prior anatomici generalizzano?
# Il numero che conta per il paper e' l'AVR su OCHuman a parita' di AP.
import torch, os
import evaluation as ev

# --- Path del checkpoint candidato (L1 boost 5x, epoca 1) ---
CAND = os.path.abspath(os.path.join(CHECKPOINT_DIR, "..", "checkpoints_L1_boost5", "epoch_01.pth"))
assert os.path.exists(CAND), f"checkpoint non trovato: {CAND}"
print("Candidato STL:", CAND)

OUT = os.path.abspath(os.path.join(CHECKPOINT_DIR, "..", "eval_confirm"))
os.makedirs(OUT, exist_ok=True)

def load(path):
    m = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
    m.load_state_dict(torch.load(path, map_location=device))
    m.eval()
    return m

# --- BASELINE ---
print("\n=== BASELINE (best.pth) ===")
m_base = load(BEST_PTH)
print("-- COCO val --")
coco_base, avr_coco_base = ev.evaluate_on_coco_val(
    m_base, val_samples, device, results_path=f"{OUT}/coco_base.json")
print("-- OCHuman (zero-shot) --")
oc_base, avr_oc_base = ev.evaluate_on_ochuman(
    m_base, device, results_path=f"{OUT}/oc_base.json")

# --- STL (epoch_01) ---
print("\n=== STL (epoch_01, L1 boost5) ===")
m_stl = load(CAND)
print("-- COCO val --")
coco_stl, avr_coco_stl = ev.evaluate_on_coco_val(
    m_stl, val_samples, device, results_path=f"{OUT}/coco_stl.json")
print("-- OCHuman (zero-shot) --")
oc_stl, avr_oc_stl = ev.evaluate_on_ochuman(
    m_stl, device, results_path=f"{OUT}/oc_stl.json")

# --- Tabella di confronto ---
LABELS = ["AP", "AP.50", "AP.75", "AP_M", "AP_L", "AR", "AR.50", "AR.75", "AR_M", "AR_L"]
print("\n" + "=" * 70)
print("CONFRONTO BASELINE vs STL")
print("=" * 70)
print(f"{'':14}{'COCO base':>11}{'COCO STL':>11}{'  ':>3}{'OC base':>11}{'OC STL':>11}")
print("-" * 70)
for i, lab in enumerate(LABELS):
    cb, cs = coco_base.stats[i], coco_stl.stats[i]
    ob, os_ = oc_base.stats[i], oc_stl.stats[i]
    print(f"{lab:14}{cb:>11.4f}{cs:>11.4f}{'  ':>3}{ob:>11.4f}{os_:>11.4f}")

print("-" * 70)
# AVR
acb, acs = avr_coco_base['AVR_pose_rate'], avr_coco_stl['AVR_pose_rate']
aob, aos = avr_oc_base['AVR_pose_rate'], avr_oc_stl['AVR_pose_rate']
print(f"{'AVR rate':14}{acb:>11.4f}{acs:>11.4f}{'  ':>3}{aob:>11.4f}{aos:>11.4f}")
mcb, mcs = avr_coco_base['AVR_mean_violations'], avr_coco_stl['AVR_mean_violations']
mob, mos = avr_oc_base['AVR_mean_violations'], avr_oc_stl['AVR_mean_violations']
print(f"{'AVR mean':14}{mcb:>11.4f}{mcs:>11.4f}{'  ':>3}{mob:>11.4f}{mos:>11.4f}")

# per-categoria
print("-" * 70)
print("AVR per-categoria (rate):")
for cat in ['bone_ratio', 'joint_angle', 'collapse']:
    pcb = avr_coco_base['per_category'][cat]; pcs = avr_coco_stl['per_category'][cat]
    pob = avr_oc_base['per_category'][cat];  pos = avr_oc_stl['per_category'][cat]
    print(f"  {cat:12}{pcb:>11.4f}{pcs:>11.4f}{'  ':>3}{pob:>11.4f}{pos:>11.4f}")

# --- Verdetto ---
print("\n" + "=" * 70)
print("VERDETTO")
print("=" * 70)
d_ap_coco = coco_stl.stats[0] - coco_base.stats[0]
d_ap_oc   = oc_stl.stats[0] - oc_base.stats[0]
d_avr_coco = acs - acb
d_avr_oc   = aos - aob
print(f"  COCO  : AP {coco_base.stats[0]:.4f}->{coco_stl.stats[0]:.4f} ({d_ap_coco:+.4f})  "
      f"AVR {acb:.4f}->{acs:.4f} ({d_avr_coco:+.4f})")
print(f"  OCHuman: AP {oc_base.stats[0]:.4f}->{oc_stl.stats[0]:.4f} ({d_ap_oc:+.4f})  "
      f"AVR {aob:.4f}->{aos:.4f} ({d_avr_oc:+.4f})")
print("-" * 70)
ok_coco = (d_avr_coco < 0) and (d_ap_coco > -0.010)
ok_oc   = (d_avr_oc < 0) and (d_ap_oc > -0.010)
if ok_coco and ok_oc:
    print("  RISULTATO SOLIDO: AVR scende su ENTRAMBI i dataset con AP preservata.")
    print("  -> Risultato pubblicabile. La generalizzazione zero-shot su OCHuman")
    print("     e' la conferma della tesi (prior anatomici dataset-independent).")
elif ok_oc:
    print("  AVR scende su OCHuman (la tesi regge) ma su COCO meno netto. Comunque")
    print("  un buon risultato: il valore e' la generalizzazione zero-shot.")
elif ok_coco:
    print("  AVR scende su COCO ma NON generalizza a OCHuman. Risultato debole per")
    print("  la tesi: i prior non si trasferiscono. Rivedere.")
else:
    print("  epoch_01 non migliora come sperato in valutazione completa. Controlla")
    print("  se un'altra epoca (epoch_02) e' un candidato migliore.")
print("=" * 70)

In [ ]:
# === 4b-SANITY-v2. 3 epoche + ribilanciamento lambda verso il KPI AVR ===
# Diagnosi dal sanity v1: order dominava la loss (0.80) ma NON e' una categoria
# dell'AVR; bone/angle (che muovono l'AVR) erano schiacciati; collapse gia' a 0.
# Qui: (1) calibro come prima, (2) DECLASSO order a un lambda piccolo fisso,
# (3) giro 3 epoche tracciando AVR e le sue 3 sotto-categorie per vedere la
# traiettoria, non un punto solo. order resta come prior strutturale a basso
# peso (e' un inductive bias sul kinematic tree, non un target del KPI).
import torch, os, time
from tqdm.auto import tqdm
import evaluation as ev
from stl import SkeletalTopologyLoss, calibrate_lambdas

# --- Baseline di riferimento ---
BASE_AP, BASE_AVR = 0.498, 0.306
AP_FLOOR = BASE_AP - 0.010
N_SANITY_EPOCHS = 3
ORDER_LAMBDA_FIXED = 1e-4   # order declassato: prior strutturale, non KPI

# --- best.pth dal config (auto-detect) ---
_best = BEST_PTH
assert os.path.exists(_best), f"best.pth non trovato: {_best}"
print("Baseline checkpoint:", _best, "| Ambiente:", ENV_NAME)

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(_best, map_location=device))
print("Pesi baseline caricati")

LR_STL = STL_FINE_TUNE_LR
criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA,
)

# --- Calibrazione standard (calibra tutti e 4) ---
print("Calibrazione lambda (gradient-norm)...")
lam = calibrate_lambdas(
    criterion, model, train_loader, device,
    target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True,
)

# --- RIBILANCIAMENTO: declasso order ---
# order non e' nel KPI AVR. Lo fisso a un valore piccolo cosi non monopolizza
# il gradiente; bone/angle/collapse restano coi loro lambda calibrati.
lam_before_order = criterion.lambda_order
criterion.lambda_order = ORDER_LAMBDA_FIXED
print(f"\nRIBILANCIAMENTO: order {lam_before_order:.5f} -> {ORDER_LAMBDA_FIXED:.5f} (declassato)")
print(f"Lambda finali: bone={criterion.lambda_bone:.5f} angle={criterion.lambda_angle:.5f} "
      f"order={criterion.lambda_order:.5f} collapse={criterion.lambda_collapse:.5f}")

optimizer = torch.optim.Adam(model.parameters(), lr=LR_STL)
optimizer.zero_grad(set_to_none=True)

SANITY_DIR = os.path.join(CHECKPOINT_DIR, "..", "checkpoints_sanity_v2")
SANITY_DIR = os.path.abspath(SANITY_DIR)
os.makedirs(SANITY_DIR, exist_ok=True)

print(f"\n{'ep':>3} {'AP':>7} {'AR':>7} {'AVR':>7} {'bone':>7} {'angle':>7} {'colla':>7}  "
      f"{'L_bone':>8} {'L_ang':>8} {'L_ord':>8} {'L_col':>8}")
print("-" * 92)

# --- Baseline AVR per-categoria (epoca 0): valuto PRIMA di allenare ---
model.eval()
_, avr0 = ev.evaluate_on_coco_val(
    model, val_samples, device,
    results_path=f"{SANITY_DIR}/pred_e00.json")
pc0 = avr0['per_category']
print(f"{'0':>3} {'-':>7} {'-':>7} {avr0['AVR_pose_rate']:>7.4f} "
      f"{pc0['bone_ratio']:>7.4f} {pc0['joint_angle']:>7.4f} {pc0['collapse']:>7.4f}  "
      f"{'(baseline pre-training)':>0}")

history = []
for epoch in range(1, N_SANITY_EPOCHS + 1):
    t0 = time.time()
    model.train()
    sums = {k: 0.0 for k in ['heatmap','bone','angle','order','collapse','total']}
    for imgs, hms, w in tqdm(train_loader, desc=f"ep {epoch}", leave=False):
        imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss, terms = criterion(out, hms, w)
        loss.backward()
        optimizer.step()
        for k in sums: sums[k] += terms[k] * imgs.size(0)
    n = len(train_loader.dataset)
    for k in sums: sums[k] /= n

    torch.save(model.state_dict(), f"{SANITY_DIR}/epoch_{epoch:02d}.pth")

    model.eval()
    coco_eval, avr = ev.evaluate_on_coco_val(
        model, val_samples, device,
        results_path=f"{SANITY_DIR}/pred_e{epoch:02d}.json")
    ap = float(coco_eval.stats[0]); ar = float(coco_eval.stats[5])
    pc = avr['per_category']; avr_rate = avr['AVR_pose_rate']

    print(f"{epoch:>3} {ap:>7.4f} {ar:>7.4f} {avr_rate:>7.4f} "
          f"{pc['bone_ratio']:>7.4f} {pc['joint_angle']:>7.4f} {pc['collapse']:>7.4f}  "
          f"{sums['bone']:>8.4f} {sums['angle']:>8.4f} {sums['order']:>8.4f} {sums['collapse']:>8.4f}  "
          f"{time.time()-t0:.0f}s")
    history.append({'epoch': epoch, 'AP': ap, 'AVR': avr_rate, **{f'pc_{k}': v for k,v in pc.items()}})

# --- Verdetto sulla TRAIETTORIA ---
print("\n" + "=" * 60)
print("VERDETTO (traiettoria su 3 epoche)")
print("=" * 60)
avr_traj = [avr0['AVR_pose_rate']] + [h['AVR'] for h in history]
ap_traj  = [h['AP'] for h in history]
print(f"  AVR: " + " -> ".join(f"{a:.4f}" for a in avr_traj))
print(f"  AP : baseline {BASE_AP:.4f}, poi " + " -> ".join(f"{a:.4f}" for a in ap_traj))

ap_ok = all(a >= AP_FLOOR for a in ap_traj)
avr_decreasing = avr_traj[-1] < avr_traj[0]
avr_below_base = history[-1]['AVR'] < BASE_AVR

print("-" * 60)
print(f"  AP regge (tutte le epoche >= {AP_FLOOR:.4f}): {'PASS' if ap_ok else 'FAIL'}")
print(f"  AVR in calo (E3 < E0):                      {'PASS' if avr_decreasing else 'FAIL'}")
print(f"  AVR sotto baseline (E3 < {BASE_AVR}):         {'PASS' if avr_below_base else 'FAIL'}")
print("-" * 60)
if ap_ok and avr_decreasing:
    print("  ESITO: traiettoria SCENDE con AP intatto -> lancia il run completo")
    print("         (eventualmente con piu' epoche per consolidare).")
else:
    print("  ESITO: l'AVR non risponde nemmeno declassando order.")
    print("         Prossimo sospetto: il gate STL (target_weight) vs AVR (MIN_CONF).")
    print("         Vedi quale sotto-categoria non scende (bone/angle/collapse).")
print("=" * 60)

In [ ]:
# === 4b. Fine-tuning con STL (calibrazione lambda integrata + selezione su AP COCO val) ===
import torch, os, time
from tqdm.auto import tqdm
import evaluation as ev
from stl import SkeletalTopologyLoss, calibrate_lambdas

# --- BEST_PTH portabile: rispetta quello del config se esiste, altrimenti
#     cade sul path Kaggle del dataset condiviso. Cosi la stessa cella gira
#     sia in locale (config locale definisce BEST_PTH) sia su Kaggle. ---
try:
    _best = BEST_PTH
except NameError:
    _best = None
if not _best or not os.path.exists(_best):
    _best = "/kaggle/input/datasets/messinaalberto/pose-baseline-checkpoint/best.pth"
assert os.path.exists(_best), f"best.pth non trovato: {_best}"
print("Baseline checkpoint:", _best)

model = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model.load_state_dict(torch.load(_best, map_location=device))
print('Pesi baseline caricati')

# --- Iperparametri fine-tuning (tutti da config.py) ---
NUM_EPOCHS_STL = STL_NUM_EPOCHS
LR_STL = STL_FINE_TUNE_LR   # 3e-5: evita catastrophic forgetting (era 1e-4)

# --- Calibrazione lambda INTEGRATA (non dipende piu' dalla cella 3c) ---
# Misura ||grad L_t / d(final_layer)|| per ogni termine non pesato e per la
# heatmap loss, poi lambda_t = STL_TARGET_FRAC * g_hm / (g_t + eps), con spread
# clamp (max 20x) per stabilita'. Vedi stl.py::calibrate_lambdas.
# La eseguo SUL MODELLO BASELINE appena caricato, prima del fine-tuning, cosi
# i lambda riflettono lo stato da cui partiamo. Se lam_suggest esiste gia'
# (cella 3c eseguita a mano), lo rispetto e salto la ricalibrazione.
criterion = SkeletalTopologyLoss(
    heatmap_criterion=train.WeightedMSELoss(),
    lambda_bone=1.0, lambda_angle=1.0, lambda_order=1.0, lambda_collapse=1.0,
    beta=STL_BETA,
)

try:
    lam = lam_suggest  # gia' calibrato dalla cella 3c
    criterion.lambda_bone     = lam['bone']
    criterion.lambda_angle    = lam['angle']
    criterion.lambda_order    = lam['order']
    criterion.lambda_collapse = lam['collapse']
    print("Uso lambda gia' calibrati dalla cella 3c")
except NameError:
    print('Calibrazione lambda in corso (gradient-norm, su baseline)...')
    lam = calibrate_lambdas(
        criterion, model, train_loader, device,
        target_frac=STL_TARGET_FRAC, n_batches=4, verbose=True,
    )
    # calibrate_lambdas aggiorna criterion in-place e ritorna i lambda.

print(f"  lambda finali: bone={criterion.lambda_bone:.5f} "
      f"angle={criterion.lambda_angle:.5f} order={criterion.lambda_order:.5f} "
      f"collapse={criterion.lambda_collapse:.5f}")

# IMPORTANTE: calibrate_lambdas lascia gradienti sporchi sul modello.
# Azzero prima di creare l'optimizer e iniziare il training.
optimizer = torch.optim.Adam(model.parameters(), lr=LR_STL)
optimizer.zero_grad(set_to_none=True)

# milestone a 70% delle epoche: si scala automaticamente se cambi NUM_EPOCHS_STL
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[round(0.7 * NUM_EPOCHS_STL)], gamma=0.1)

STL_CKPT_DIR = '/kaggle/working/checkpoints_stl'
os.makedirs(STL_CKPT_DIR, exist_ok=True)

# --- Criterio di selezione: AP COCO val, NON val_loss ---
# val_loss include la STL: minimizzarla premia un modello con STL bassa anche
# se l'AP e' peggiorata. L'obiettivo del progetto e' abbassare AVR SENZA
# distruggere AP, quindi seleziono sull'AP (stats[0] di pycocotools).
# Salvo comunque ogni epoca, cosi la scelta finale resta ispezionabile a mano.
best_ap = -1.0
history = []

for epoch in range(1, NUM_EPOCHS_STL + 1):
    t0 = time.time()
    model.train()
    sums = {k: 0.0 for k in ['heatmap', 'bone', 'angle', 'order', 'collapse', 'total']}
    for imgs, hms, w in tqdm(train_loader, desc=f'train {epoch}', leave=False):
        imgs, hms, w = imgs.to(device), hms.to(device), w.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss, terms = criterion(out, hms, w)
        loss.backward()
        optimizer.step()
        for k in sums: sums[k] += terms[k] * imgs.size(0)
    n = len(train_loader.dataset)
    for k in sums: sums[k] /= n

    # Salvo SEMPRE il checkpoint di questa epoca (scelta finale ispezionabile)
    torch.save(model.state_dict(), f'{STL_CKPT_DIR}/epoch_{epoch:02d}.pth')

    # Valuto AP+AVR su COCO val per la selezione
    model.eval()
    coco_eval, avr = ev.evaluate_on_coco_val(
        model, val_samples, device,
        results_path=f'{STL_CKPT_DIR}/coco_val_pred_e{epoch:02d}.json')
    ap = float(coco_eval.stats[0])      # AP @ IoU=0.50:0.95
    ar = float(coco_eval.stats[5])      # AR @ IoU=0.50:0.95
    avr_rate = avr['AVR_pose_rate']
    scheduler.step()
    lr = optimizer.param_groups[0]['lr']

    print(f"E{epoch:02d} tot:{sums['total']:.4f} hm:{sums['heatmap']:.4f} "
          f"bone:{sums['bone']:.4f} ang:{sums['angle']:.4f} "
          f"ord:{sums['order']:.4f} col:{sums['collapse']:.4f} "
          f"| AP:{ap:.4f} AR:{ar:.4f} AVR:{avr_rate:.4f} "
          f"lr:{lr:.0e} {time.time()-t0:.0f}s")
    history.append({'epoch': epoch, 'AP': ap, 'AR': ar, 'AVR': avr_rate,
                    **{f'L_{k}': sums[k] for k in sums}})

    if ap > best_ap:
        best_ap = ap
        torch.save(model.state_dict(), f'{STL_CKPT_DIR}/best_stl.pth')
        print(f'  -> nuovo best STL (AP {ap:.4f}, AVR {avr_rate:.4f})')

print('\nFine-tuning completato.')
print('Storia per-epoca (scegli a mano il trade-off AP/AVR che preferisci):')
for h in history:
    print(f"  E{h['epoch']:02d}  AP {h['AP']:.4f}  AR {h['AR']:.4f}  AVR {h['AVR']:.4f}")

# NB: best_stl.pth = miglior AP. Se un'epoca con AP di poco inferiore ha un AVR
# molto migliore, e' un trade-off legittimo per il paper: i checkpoint epoch_NN.pth
# sono tutti su disco per ricaricare quello che preferisci.


In [ ]:
# === 5. Valutazione: confronto baseline vs STL ===
import torch

model_base = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model_base.load_state_dict(torch.load(BEST_PTH, map_location=device))
print("=== BASELINE ===")
coco_base, avr_coco_base = ev.evaluate_on_coco_val(model_base, val_samples, device)
oc_base, avr_oc_base = ev.evaluate_on_ochuman(model_base, device)

model_stl = network.PoseMobileNet(NUM_KEYPOINTS, INPUT_SIZE, pretrained=False).to(device)
model_stl.load_state_dict(torch.load(f'{STL_CKPT_DIR}/best_stl.pth', map_location=device))
print("\n=== STL ===")
coco_stl, avr_coco_stl = ev.evaluate_on_coco_val(model_stl, val_samples, device,
    results_path='/kaggle/working/coco_val_pred_stl.json')
oc_stl, avr_oc_stl = ev.evaluate_on_ochuman(model_stl, device,
    results_path='/kaggle/working/ochuman_pred_stl.json')

LABELS = ["AP","AP.50","AP.75","AP_M","AP_L","AR","AR.50","AR.75","AR_M","AR_L"]
header = f"{'':15}{'COCO base':>10}{'COCO STL':>10}{'':3}{'OC base':>10}{'OC STL':>10}"
print(f"\n{header}")
print("-" * 58)
for i, lab in enumerate(LABELS):
    cb = coco_base.stats[i]; cs = coco_stl.stats[i]
    ob = oc_base.stats[i]; os_ = oc_stl.stats[i]
    print(f"{lab:15}{cb:10.4f}{cs:10.4f}{'':3}{ob:10.4f}{os_:10.4f}")
ab = avr_coco_base['AVR_pose_rate']; as_ = avr_coco_stl['AVR_pose_rate']
ob = avr_oc_base['AVR_pose_rate']; os_ = avr_oc_stl['AVR_pose_rate']
print(f"\n{'AVR rate':15}{ab:10.4f}{as_:10.4f}{'':3}{ob:10.4f}{os_:10.4f}")
ab = avr_coco_base['AVR_mean_violations']; as_ = avr_coco_stl['AVR_mean_violations']
ob = avr_oc_base['AVR_mean_violations']; os_ = avr_oc_stl['AVR_mean_violations']
print(f"{'AVR mean':15}{ab:10.4f}{as_:10.4f}{'':3}{ob:10.4f}{os_:10.4f}")